# Encoder–decoder LoRMA + DoRA for legal NER
Set the exact Phase-1 checkpoint tag and matching tokenizer bucket path before running.
Encoder reads each chunk bidirectionally; causal decoder reads those same observed tokens and cross-attends to encoder memory. Decoder features use the original mean word pooling and BIO classifier.
LoRMA + DoRA adapters cover all encoder, decoder and cross-attention projections. Last two encoder and decoder blocks are also fully tuned by default; set FULL_TUNE_LAST_BLOCKS=0 for pure PEFT.
Preprocessing and chunking are unchanged. No LSTM or CRF is used in this experiment. Train 20 epochs and restore best validation model.
Validation retains the original small split and reconstructed BIO gold; its F1 is not a verified exact-offset competition score.
The original generation/cache helpers are inherited but unused for NER.


In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 6.5 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train_200M.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val_200M.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [8]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [9]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

## Checkpoint related functions

In [10]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [11]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        upload_checkpoint(tmp_dir,tag)

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        os.makedirs(checkpoint_dir, exist_ok=True)

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [12]:
PHASE1_TAG = "encoder_decoder_mixed"
ENCDEC_TOKENIZER_PATH = "tokenizer_encoder_decoder/tokenizer.json"
if ENCDEC_TOKENIZER_PATH.startswith("SET_"):
    raise ValueError("Set ENCDEC_TOKENIZER_PATH to the exact tokenizer used for this Phase-1 checkpoint")


In [13]:
from pathlib import Path
from tokenizers import Tokenizer

LOCAL_DIR = Path("tokenizer_encoder_decoder")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TOKENIZER = LOCAL_DIR / "tokenizer.json"

# Download exact tokenizer from bucket
download_from_bucket(
    ENCDEC_TOKENIZER_PATH,
    str(LOCAL_TOKENIZER)
)

# Load tokenizer
tokenizer = Tokenizer.from_file(
    str(LOCAL_TOKENIZER)
)

print("Tokenizer loaded successfully")
print("Vocab size:", tokenizer.get_vocab_size())

(…)tokenizer_encoder_decoder/tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

Downloaded: tokenizer_encoder_decoder/tokenizer.json -> tokenizer_encoder_decoder/tokenizer.json
Tokenizer loaded successfully
Vocab size: 33178


In [14]:
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# --------------------------------------------------
# Download
# --------------------------------------------------

fold1_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-text.json",
    repo_type="dataset"
)

fold1_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-gold.json",
    repo_type="dataset"
)

fold2_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-text.json",
    repo_type="dataset"
)

fold2_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-gold.json",
    repo_type="dataset"
)

labels_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/labels.txt",
    repo_type="dataset"
)

# --------------------------------------------------
# Load JSON
# --------------------------------------------------

with open(fold1_text_path, "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(fold1_gold_path, "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)

with open(fold2_text_path, "r", encoding="utf-8") as f:
    fold2_text = json.load(f)

with open(fold2_gold_path, "r", encoding="utf-8") as f:
    fold2_gold = json.load(f)

# --------------------------------------------------
# Load labels
# --------------------------------------------------

with open(labels_path, "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

print("Fold 1 documents:", len(fold1_text))
print("Fold 1 annotations:", len(fold1_gold))

print("Fold 2 documents:", len(fold2_text))
print("Fold 2 annotations:", len(fold2_gold))

print("Labels:")
print(labels)

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Fold 1 documents: 35
Fold 1 annotations: 35
Fold 2 documents: 35
Fold 2 annotations: 35
Labels:
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [15]:
VOCAB_SIZE = tokenizer.get_vocab_size()

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 5.0

ARCH_TRAIN_TOKENS = 100_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [16]:
VOCAB_SIZE

33178

## Model - A1 

In [17]:
VOCAB_SIZE = tokenizer.get_vocab_size()
DIM=640
ENCODER_LAYERS=6
DECODER_LAYERS=6
N_HEADS=16
N_KV_HEADS=4
FF_DIM=1728
MAX_ENCODER_LEN=512
MAX_DECODER_LEN=512
RMS_NORM_EPS=1e-6
ROPE_BASE=10000
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
        key_mask=None,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]

            cos, sin = self.rope(
                current_len
            )

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        attention_mask = None
        if key_mask is not None:
            attention_mask = key_mask[:, None, None, :].bool()
            if is_causal:
                causal_mask = torch.ones(T, k.size(2), device=x.device, dtype=torch.bool).tril()
                attention_mask = attention_mask & causal_mask[None, None]
            is_causal = False

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = (k, v)

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_seq_len,
    ):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_seq_len,
            causal=False,
            use_rope=True,
        )

        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(self, x, key_mask=None):

        # Pre-norm self-attention
        x = x + self.self_attn(
            self.norm1(x), key_mask=key_mask
        )[0]

        # Pre-norm FFN
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
        key_mask=None,
        encoder_mask=None,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
            key_mask=key_mask,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
            key_mask=encoder_mask,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_seq_len=max_encoder_len,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=max_decoder_len,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================


class LormaDoRALinear(nn.Module):
    def __init__(
        self,
        original_linear,
        r=16,
        alpha=32,
        dropout=0.05,
        do_pi=True,
        eps=1e-8,
    ):
        super().__init__()

        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.do_pi = do_pi
        self.eps = eps

        self.dropout = nn.Dropout(dropout)

        # Preserve the original pretrained linear layer.
        self.original_linear = original_linear

        # Freeze pretrained weights and bias.
        for param in self.original_linear.parameters():
            param.requires_grad = False

        device = original_linear.weight.device
        dtype = original_linear.weight.dtype

        # LoRMA pre-transformation:
        # B: [out_features, r]
        # A: [r, out_features]
        self.B = nn.Parameter(
            torch.empty(
                self.out_features, r,
                device=device, dtype=dtype,
            )
        )
        self.A = nn.Parameter(
            torch.empty(
                r, self.out_features,
                device=device, dtype=dtype,
            )
        )

        # Initialize the transformation near identity.
        # BA = I when r >= out_features and the initialization
        # is constructed as below; otherwise it is low rank.
        nn.init.zeros_(self.B)
        nn.init.zeros_(self.A)

        with torch.no_grad():
            # This construction makes B @ A = I for any r >= 1:
            # B[:, 0] = 1, A[0, :] = 1 does NOT produce identity.
            # Instead, use a rectangular identity factorization
            # when rank is sufficient.
            k = min(r, self.out_features)
            self.B[:k, :k] = torch.eye(
                k, device=device, dtype=dtype
            )
            self.A[:k, :k] = torch.eye(
                k, device=device, dtype=dtype
            )

        # DoRA-style row magnitudes, initialized from pretrained W.
        initial_magnitude = torch.linalg.vector_norm(
            original_linear.weight.detach().float(),
            ord=2,
            dim=1,
        ).to(device=device, dtype=dtype)

        self.magnitude = nn.Parameter(initial_magnitude)

    def _shift(self, matrix):
        """
        Row-dependent cyclic column shift.
        Matches the shift convention used in our LoRMA implementation.
        """
        n_rows, n_cols = matrix.shape

        row_indices = torch.arange(
            n_rows, device=matrix.device
        ).unsqueeze(1)

        col_indices = torch.arange(
            n_cols, device=matrix.device
        ).unsqueeze(0)

        indices = (col_indices - row_indices) % n_cols

        return torch.gather(matrix, 1, indices)

    def effective_weight(self):
        W = self.original_linear.weight
    
        P_delta = self.B @ self.A
    
        if self.do_pi:
            P_delta = self._shift(P_delta)
    
        identity = torch.eye(
            self.out_features,
            device=W.device,
            dtype=W.dtype,
        )
    
        P = identity + self.scaling * P_delta
        transformed_W = P @ W
    
        row_norms = torch.linalg.vector_norm(
            transformed_W.float(),
            ord=2,
            dim=1,
            keepdim=True,
        ).to(dtype=transformed_W.dtype)
    
        normalized_W = transformed_W / row_norms.clamp_min(self.eps)
    
        return self.magnitude.unsqueeze(1) * normalized_W

    def forward(self, x):
        W_eff = self.effective_weight()

        return F.linear(
            self.dropout(x),
            W_eff,
            self.original_linear.bias,
        )


class EncoderDecoderForNER(EncoderDecoderLM):
    def __init__(self, num_labels=25, **kwargs):
        super().__init__(**kwargs)
        self.ner_head = nn.Linear(self.dim, num_labels)

    def forward(self, input_ids, word_groups):
        lengths = torch.tensor(
            [max(index for group in groups for index in group) + 1 for groups in word_groups],
            device=input_ids.device,
        )
        mask = torch.arange(input_ids.size(1), device=input_ids.device)[None] < lengths[:, None]
        memory = self.token_embedding(input_ids)
        for block in self.encoder_blocks:
            memory = block(memory, key_mask=mask)
        memory = self.encoder_norm(memory)
        # NER classifies observed tokens, so decoder inputs are not shifted.
        hidden = self.token_embedding(input_ids)
        for block in self.decoder_blocks:
            hidden, _ = block(hidden, memory, key_mask=mask, encoder_mask=mask)
        hidden = self.decoder_norm(hidden)
        words = [torch.stack([hidden[b, group].mean(0) for group in groups])
                 for b, groups in enumerate(word_groups)]
        padded = nn.utils.rnn.pad_sequence(words, batch_first=True)
        return self.ner_head(padded)


In [18]:
# Must match your Phase-1 encoder-decoder checkpoint exactly.
DIM = 640
ENCODER_LAYERS = 6
DECODER_LAYERS = 6
N_HEADS = 16
N_KV_HEADS = 4
FF_DIM = 1728
MAX_ENCODER_LEN = 512
MAX_DECODER_LEN = 512  # Extend nonpersistent RoPE buffers for NER chunks.
seq_len = 512
MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, dim=DIM,
    encoder_layers=ENCODER_LAYERS, decoder_layers=DECODER_LAYERS,
    n_heads=N_HEADS, n_kv_heads=N_KV_HEADS, ff_dim=FF_DIM,
    max_encoder_len=MAX_ENCODER_LEN, max_decoder_len=MAX_DECODER_LEN)


In [19]:
teacher_base = EncoderDecoderLM(**MODEL_KWARGS).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in teacher_base.parameters())/1e6:.2f}M")


Parameters: 79.50M


In [20]:

teacher_optimizer_dummy = torch.optim.AdamW(teacher_base.parameters(), lr=LEARNING_RATE)
teacher_state = load_checkpoint(PHASE1_TAG, teacher_base, teacher_optimizer_dummy)
print("Loaded encoder-decoder Phase-1 checkpoint")


(…)eckpoints/encoder_decoder_mixed/model.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

Downloaded: checkpoints/encoder_decoder_mixed/model.pt -> /tmp/tmpea3di4vc/checkpoints/encoder_decoder_mixed/model.pt


(…)oints/encoder_decoder_mixed/optimizer.pt:   0%|          | 0.00/636M [00:00<?, ?B/s]

Downloaded: checkpoints/encoder_decoder_mixed/optimizer.pt -> /tmp/tmpea3di4vc/checkpoints/encoder_decoder_mixed/optimizer.pt


(…)oints/encoder_decoder_mixed/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/encoder_decoder_mixed/rng_state.pt -> /tmp/tmpea3di4vc/checkpoints/encoder_decoder_mixed/rng_state.pt


(…)ncoder_decoder_mixed/training_state.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Downloaded: checkpoints/encoder_decoder_mixed/training_state.json -> /tmp/tmpea3di4vc/checkpoints/encoder_decoder_mixed/training_state.json
Loaded encoder-decoder Phase-1 checkpoint


In [21]:
del teacher_optimizer_dummy
del teacher_state
import gc
gc.collect()

140

In [22]:
labels

['APP',
 'RESP',
 'A.COUNSEL',
 'R.COUNSEL',
 'JUDGE',
 'WIT',
 'AUTH',
 'COURT',
 'STAT',
 'PREC',
 'DATE',
 'CASENO']

In [23]:
teacher = EncoderDecoderForNER(num_labels=25, **MODEL_KWARGS).to(DEVICE)


In [24]:
missing, unexpected = teacher.load_state_dict(teacher_base.state_dict(), strict=False)
assert set(missing) == {"ner_head.weight", "ner_head.bias"}, missing
assert not unexpected, unexpected
print("Transferred all Phase-1 backbone weights; initialized new BIO classifier")


Transferred all Phase-1 backbone weights; initialized new BIO classifier


In [25]:
attention_modules = [b.self_attn for b in teacher.encoder_blocks]
attention_modules += [a for b in teacher.decoder_blocks for a in (b.self_attn, b.cross_attn)]
for attention in attention_modules:
    for name in ("q_proj", "k_proj", "v_proj", "o_proj"):
        setattr(attention, name, LormaDoRALinear(getattr(attention, name), r=16, alpha=32, dropout=0.05, do_pi=True))


In [26]:
for p in teacher.parameters():
    p.requires_grad = False

In [27]:
for module in teacher.modules():
    if isinstance(module, LormaDoRALinear):
        module.A.requires_grad = True
        module.B.requires_grad = True
        module.magnitude.requires_grad = True
for parameter in teacher.ner_head.parameters():
    parameter.requires_grad = True


In [28]:
# Preserve the previous hybrid strategy: adapters plus full tuning of final blocks.
# Set to 0 for pure LoRMA + DoRA with only the classifier additionally trained.
FULL_TUNE_LAST_BLOCKS = 2
if FULL_TUNE_LAST_BLOCKS:
    for block in list(teacher.encoder_blocks[-FULL_TUNE_LAST_BLOCKS:]) + list(teacher.decoder_blocks[-FULL_TUNE_LAST_BLOCKS:]):
        for parameter in block.parameters():
            parameter.requires_grad = True


In [29]:
# Entity labels from labels.txt
print(labels)

BIO_LABELS = ["O"]

for entity in labels:
    BIO_LABELS.append(f"B-{entity}")
    BIO_LABELS.append(f"I-{entity}")

label2id = {
    label: i
    for i, label in enumerate(BIO_LABELS)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Number of BIO labels:", len(BIO_LABELS))
print(label2id)

['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of BIO labels: 25
{'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4, 'B-A.COUNSEL': 5, 'I-A.COUNSEL': 6, 'B-R.COUNSEL': 7, 'I-R.COUNSEL': 8, 'B-JUDGE': 9, 'I-JUDGE': 10, 'B-WIT': 11, 'I-WIT': 12, 'B-AUTH': 13, 'I-AUTH': 14, 'B-COURT': 15, 'I-COURT': 16, 'B-STAT': 17, 'I-STAT': 18, 'B-PREC': 19, 'I-PREC': 20, 'B-DATE': 21, 'I-DATE': 22, 'B-CASENO': 23, 'I-CASENO': 24}


In [30]:
def prepare_lner_data(text_data, gold_data, tokenizer):

    processed_data = {}

    for doc_id, text in text_data.items():

       

        # --------------------------------------------------
        # 1. Tokenize
        # --------------------------------------------------

        encoding = tokenizer.encode(text)

        tokens = encoding.tokens
        input_ids = encoding.ids
        offsets = encoding.offsets

        # --------------------------------------------------
        # 2. Group tokens
        #
        # Rule:
        #   Ġ  -> new word
        #   no Ġ -> continue previous word
        # --------------------------------------------------

        word_groups = []
        current_group = []

        for i, token in enumerate(tokens):

            start, end = offsets[i]

            if start == end:
                continue

            if current_group and token.startswith("Ġ"):
                word_groups.append(current_group)
                current_group = []

            current_group.append(i)

        if current_group:
            word_groups.append(current_group)

        # --------------------------------------------------
        # 3. Correct character spans
        #
        # Raw tokenizer offsets include the whitespace
        # represented by Ġ.
        # Remove that whitespace from the beginning.
        # --------------------------------------------------

        word_spans = []

        for group in word_groups:

            start = offsets[group[0]][0]
            end = offsets[group[-1]][1]

            while start < end and text[start].isspace():
                start += 1

            word_spans.append((start, end))

        # --------------------------------------------------
        # 4. Initialize labels
        # --------------------------------------------------

        word_labels = ["O"] * len(word_groups)

        # --------------------------------------------------
        # 5. Align gold entities
        # --------------------------------------------------
        if gold_data is not None:
            gold_entities = gold_data[doc_id]
            for entity_start, entity_end, entity_type, entity_text in gold_entities:
    
                overlapping_words = []
    
                for word_idx, (word_start, word_end) in enumerate(word_spans):
    
                    if word_start < entity_end and word_end > entity_start:
                        overlapping_words.append(word_idx)
    
                if not overlapping_words:
                    continue
    
                word_labels[overlapping_words[0]] = f"B-{entity_type}"
    
                for word_idx in overlapping_words[1:]:
                    word_labels[word_idx] = f"I-{entity_type}"

        # --------------------------------------------------
        # 6. Save
        # --------------------------------------------------

        processed_data[doc_id] = {
            "input_ids": input_ids,
            "offsets": offsets,
            "word_groups": word_groups,
            "word_spans": word_spans,
            "word_labels": word_labels,
        }

    return processed_data

In [31]:
fold1_processed = prepare_lner_data(
    fold1_text,
    fold1_gold,
    tokenizer
)


In [32]:
fold2_processed = prepare_lner_data(
    fold2_text,
    fold2_gold,
    tokenizer
)

In [33]:
MAX_SEQ_LEN = 512

def make_chunks(example, max_seq_len=512):
    """
    Split one processed document into chunks at word boundaries.

    Returns list of:
        {
            "input_ids": [...],
            "word_groups": [...],
            "word_labels": [...]
        }
    """

    input_ids = example["input_ids"]
    word_groups = example["word_groups"]
    word_labels = example["word_labels"]
    word_spans = example["word_spans"]
    chunks = []

    current_words = []
    current_start = None
    current_end = None

    for word_idx, group in enumerate(word_groups):

        word_start = group[0]
        word_end = group[-1] + 1

        if current_start is None:
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]
            continue

        # Would adding this word exceed 2048?
        new_end = word_end

        if new_end - current_start > max_seq_len:

            # Save current chunk
            chunk_input_ids = input_ids[current_start:current_end]

            chunk_groups = [
                [
                    token_idx - current_start
                    for token_idx in word_groups[w]
                ]
                for w in current_words
            ]

            chunk_labels = [
                word_labels[w]
                for w in current_words
            ]
            chunk_spans = [
                word_spans[w]
                for w in current_words
            ]
            chunks.append({
                "input_ids": chunk_input_ids,
                "word_groups": chunk_groups,
                "word_labels": chunk_labels,
                "word_spans" : chunk_spans
            })

            # Start new chunk
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]

        else:
            current_end = word_end
            current_words.append(word_idx)

    # Last chunk
    if current_words:

        chunk_input_ids = input_ids[current_start:current_end]

        chunk_groups = [
            [
                token_idx - current_start
                for token_idx in word_groups[w]
            ]
            for w in current_words
        ]

        chunk_labels = [
            word_labels[w]
            for w in current_words
        ]
        chunk_spans = [
            word_spans[w]
            for w in current_words
        ]
        chunks.append({
            "input_ids": chunk_input_ids,
            "word_groups": chunk_groups,
            "word_labels": chunk_labels,
            "word_spans" : chunk_spans
        })

    return chunks

In [34]:
import random

SEED = 42
TRAIN_RATIO = 0.99
MAX_SEQ_LEN = 512

# --------------------------------------------------
# 1. Combine Fold 1 + Fold 2
# --------------------------------------------------

all_processed_data = {}

all_processed_data.update(fold1_processed)
all_processed_data.update(fold2_processed)

doc_ids = list(all_processed_data.keys())

print("Total documents:", len(doc_ids))

# --------------------------------------------------
# 2. Shuffle documents
# --------------------------------------------------

rng = random.Random(SEED)
rng.shuffle(doc_ids)

# --------------------------------------------------
# 3. 99/1 document-level split
# --------------------------------------------------

split_idx = int(TRAIN_RATIO * len(doc_ids))

train_doc_ids = doc_ids[:split_idx]
val_doc_ids = doc_ids[split_idx:]

print("Train documents:", len(train_doc_ids))
print("Validation documents:", len(val_doc_ids))

print(
    "Train ratio:",
    len(train_doc_ids) / len(doc_ids)
)

print(
    "Validation ratio:",
    len(val_doc_ids) / len(doc_ids)
)

# --------------------------------------------------
# 4. Chunk training documents
# --------------------------------------------------

train_chunks = []

for doc_id in train_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    train_chunks.extend(chunks)

# --------------------------------------------------
# 5. Chunk validation documents
# --------------------------------------------------

val_chunks = []

for doc_id in val_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    val_chunks.extend(chunks)

# --------------------------------------------------
# 6. Check
# --------------------------------------------------

print("Train chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))

assert len(train_chunks) > 0
assert len(val_chunks) > 0

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in train_chunks
)

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in val_chunks
)

Total documents: 70
Train documents: 69
Validation documents: 1
Train ratio: 0.9857142857142858
Validation ratio: 0.014285714285714285
Train chunks: 913
Validation chunks: 15


In [35]:
def collate_lner(batch, pad_token_id):

    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    max_words = max(
        len(x["word_labels"])
        for x in batch
    )

    input_ids = []
    word_groups = []
    word_labels = []

    for x in batch:

        # -------------------------
        # Token padding
        # -------------------------

        ids = x["input_ids"]

        padded = ids + [pad_token_id] * (
            max_len - len(ids)
        )

        input_ids.append(padded)

        # -------------------------
        # Word groups
        # -------------------------

        word_groups.append(
            x["word_groups"]
        )

        # -------------------------
        # Word-label padding
        # -------------------------

        labels = [
            label2id[label]
            for label in x["word_labels"]
        ]

        labels += [-100] * (
            max_words - len(labels)
        )

        word_labels.append(labels)

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long
        ),

        "word_groups": word_groups,

        "word_labels": torch.tensor(
            word_labels,
            dtype=torch.long
        ),
    }

In [36]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_chunks,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

val_loader = DataLoader(
    val_chunks,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

In [37]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher = teacher.to(device)

NUM_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 5.0
GRAD_ACCUM_STEPS = 4

criterion = nn.CrossEntropyLoss()

trainable_params = [
    p for p in teacher.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)

Trainable parameters: 20387865


In [38]:
ENTITY_LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]


In [39]:
def decode_bio_predictions(
    predictions,
    word_spans,
):
    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        predictions,
        word_spans
    ):

        # A whitespace-only BPE group can become [end, end] after
        # leading whitespace trimming. It cannot form an entity.
        if end <= start:
            if current_label is not None and current_end > current_start:
                entities.append([current_start, current_end, current_label])
            current_label = None
            current_start = None
            current_end = None
            continue

        # O
        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        # B-ENTITY
        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

            continue

        # I-ENTITY
        if label.startswith("I-"):

            entity_type = label[2:]

            # Continue current entity
            if current_label == entity_type:

                current_end = end

            # Invalid I-ENTITY -> treat as a new entity
            else:

                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    # Flush final entity
    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return [entity for entity in entities if entity[1] > entity[0]]

In [40]:
def bio_to_spans(labels, word_spans):
    """
    Convert BIO word labels + character spans into:
        [start, end, label]
    """

    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        labels,
        word_spans
    ):

        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

        elif label.startswith("I-"):

            entity_type = label[2:]

            if current_label == entity_type:

                current_end = end

            else:
                # Treat invalid I-X as a new entity
                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return entities

from tqdm import tqdm
import torch


def generate_val_predictions(
    teacher,
    all_processed_data,
    val_doc_ids,
    id2label,
    max_seq_len=512,
):
    teacher.eval()

    predicted = {}
    gold = {}

    with torch.no_grad():

        for doc_id in tqdm(
            val_doc_ids,
            desc="Validation prediction"
        ):

            example = all_processed_data[doc_id]

            chunks = make_chunks(
                example,
                max_seq_len
            )

            doc_predictions = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                doc_predictions.extend(
                    entities
                )

            predicted[doc_id] = doc_predictions

            # Gold spans are obtained directly from the
            # original gold BIO labels + character spans
            gold_labels = example["word_labels"]

            gold[doc_id] = bio_to_spans(
                gold_labels,
                example["word_spans"]
            )

    return predicted, gold

from collections import defaultdict


def evaluate_strict_ner(
    predicted,
    gold,
    labels,
):
    """
    Strict entity-level evaluation.

    Returns:
        Macro-F1
        Macro-Precision
        Macro-Recall
        per-label statistics
    """

    stats = {
        label: {
            "tp": 0,
            "fp": 0,
            "fn": 0,
        }
        for label in labels
    }

    for doc_id in gold:

        gold_entities = gold.get(
            doc_id,
            []
        )

        pred_entities = predicted.get(
            doc_id,
            []
        )

        # Exact entity tuples
        gold_set = set(
            tuple(entity)
            for entity in gold_entities
        )

        pred_set = set(
            tuple(entity)
            for entity in pred_entities
        )

        # True positives
        for entity in pred_set & gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["tp"] += 1

        # False positives
        for entity in pred_set - gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["fp"] += 1

        # False negatives
        for entity in gold_set - pred_set:

            label = entity[2]

            if label in stats:
                stats[label]["fn"] += 1

    # -------------------------
    # Per-label metrics
    # -------------------------

    per_label = {}

    macro_precision = 0.0
    macro_recall = 0.0
    macro_f1 = 0.0

    for label in labels:

        tp = stats[label]["tp"]
        fp = stats[label]["fp"]
        fn = stats[label]["fn"]

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall /
            (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }

        macro_precision += precision
        macro_recall += recall
        macro_f1 += f1

    n = len(labels)

    macro_precision /= n
    macro_recall /= n
    macro_f1 /= n

    return {
        "strict_macro_f1": macro_f1,
        "strict_macro_precision": macro_precision,
        "strict_macro_recall": macro_recall,
        "per_label": per_label,
    }

In [41]:
from collections import Counter
import torch
import torch.nn as nn

label_counts = Counter()

for chunk in train_chunks:
    for label in chunk["word_labels"]:
        label_counts[label2id[label]] += 1

counts = torch.tensor(
    [label_counts[i] for i in range(len(label2id))],
    dtype=torch.float
)

# Inverse square-root frequency to avoid excessively
# large weights for rare labels
weights = 1.0 / torch.sqrt(counts.clamp(min=1))

# Normalize so the average class weight is 1
weights = weights / weights.mean()

# Reduce the weight of O
weights[label2id["O"]] *= 0.25

# Normalize again
weights = weights / weights.mean()

class_weights = weights.to(DEVICE)



print({
    label: round(class_weights[idx].item(), 3)
    for label, idx in label2id.items()
})

{'O': 0.01, 'B-APP': 1.127, 'I-APP': 1.086, 'B-RESP': 1.164, 'I-RESP': 1.035, 'B-A.COUNSEL': 1.622, 'I-A.COUNSEL': 1.675, 'B-R.COUNSEL': 1.67, 'I-R.COUNSEL': 1.791, 'B-JUDGE': 1.453, 'I-JUDGE': 1.344, 'B-WIT': 1.657, 'I-WIT': 1.802, 'B-AUTH': 0.732, 'I-AUTH': 0.744, 'B-COURT': 0.72, 'I-COURT': 0.611, 'B-STAT': 0.65, 'I-STAT': 0.507, 'B-PREC': 0.659, 'I-PREC': 0.278, 'B-DATE': 0.581, 'I-DATE': 0.752, 'B-CASENO': 0.833, 'I-CASENO': 0.496}


In [42]:
def train_teacher(
    teacher,
    train_loader,
    val_loader,
    label2id,
    optimizer,
    trainable_params,
    tag,
    num_epochs=5,
    resume=True,
    resume_tag=None,
    patience=3,
    min_delta=0.001,
):
    device = next(teacher.parameters()).device

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100
    )

    # =========================
    # RESUME
    # =========================

    if patience < 1 or min_delta < 0:
        raise ValueError("patience must be >= 1 and min_delta must be >= 0")
    start_epoch = 0
    best_f1 = float("-inf")
    patience_reference = float("-inf")
    epochs_without_improvement = 0
    history = []
    best_tag = f"{tag}_best"
    have_best = False

    if resume and resume_tag is not None:

        state = load_checkpoint(
            tag=resume_tag,
            model=teacher,
            optimizer=optimizer,
        )

        start_epoch = state["epoch"]
        extra = state.get("extra_state") or {}
        best_f1 = extra.get("best_f1", float("-inf"))
        patience_reference = extra.get("patience_reference", best_f1)
        epochs_without_improvement = extra.get("epochs_without_improvement", 0)
        history = extra.get("history", [])
        best_tag = extra.get("best_tag", best_tag)
        have_best = extra.get("have_best", False)
        # Older checkpoints have no early-stopping state: tracking starts here.

        print(
            f"Resuming from epoch {start_epoch}"
        )

    # =========================
    # TRAINING
    # =========================

    for epoch in range(start_epoch, num_epochs):
        if epochs_without_improvement >= patience:
            print("Resumed checkpoint already reached early-stopping patience")
            break

        teacher.train()

        total_train_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}"
        )

        for step, batch in enumerate(progress):

            input_ids = batch["input_ids"].to(device)

            word_groups = batch["word_groups"]

            targets = batch["word_labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            # [B, max_words, num_labels]
            logits = teacher(
                input_ids,
                word_groups
            )

            # Flatten:
            # logits  -> [B * max_words, num_labels]
            # targets -> [B * max_words]
            loss = criterion(
                logits.view(
                    -1,
                    logits.size(-1)
                ),
                targets.view(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                GRAD_CLIP
            )

            optimizer.step()

            total_train_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        avg_train_loss = (
            total_train_loss /
            len(train_loader)
        )

        # =========================
        # VALIDATION
        # =========================

        teacher.eval()

        total_val_loss = 0.0
        
        with torch.no_grad():
        
            for batch in val_loader:
        
                input_ids = batch["input_ids"].to(device)
                word_groups = batch["word_groups"]
                targets = batch["word_labels"].to(device)
        
                logits = teacher(
                    input_ids,
                    word_groups
                )
        
                loss = criterion(
                    logits.view(-1, logits.size(-1)),
                    targets.view(-1)
                )
        
                total_val_loss += loss.item()
        
        avg_val_loss = (
            total_val_loss /
            len(val_loader)
        )
        
        # =========================
        # STRICT VALIDATION F1
        # =========================
        
        val_predicted, val_gold = generate_val_predictions(
            teacher=teacher,
            all_processed_data=all_processed_data,
            val_doc_ids=val_doc_ids,
            id2label=id2label,
            max_seq_len=MAX_SEQ_LEN,
        )
        
        val_metrics = evaluate_strict_ner(
            predicted=val_predicted,
            gold=val_gold,
            labels=ENTITY_LABELS,
        )
        
        val_f1 = val_metrics["strict_macro_f1"]
        # =========================
        # LOGGING
        # =========================

        print(
            f"\nEpoch {epoch + 1}/{num_epochs}"
        )

        print(
            f"Train Loss:    {avg_train_loss:.4f}"
        )

        print(
            f"Val Loss:      {avg_val_loss:.4f}"
        )

        print(
            f"Val Strict F1:  {val_f1:.4f}"
        )

        # Save any strict F1 improvement, even if it is smaller than min_delta.
        improved = val_f1 > best_f1
        if improved:
            best_f1 = val_f1
            have_best = True

        # min_delta controls patience resets, not which checkpoint is best.
        if val_f1 > patience_reference + min_delta:
            patience_reference = val_f1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_strict_macro_f1": val_f1,
        })
        extra = {
            "best_f1": best_f1,
            "patience_reference": patience_reference,
            "epochs_without_improvement": epochs_without_improvement,
            "best_tag": best_tag,
            "have_best": have_best,
            "history": history,
        }
        if improved:
            save_checkpoint(
                tag=best_tag, model=teacher, optimizer=optimizer,
                epoch=epoch + 1, extra_state=extra,
            )
            print(f"New best checkpoint: {best_tag} (F1={best_f1:.4f})")

        # Keep latest state separately so an interrupted run can resume.
        save_checkpoint(
            tag=tag, model=teacher, optimizer=optimizer,
            epoch=epoch + 1, extra_state=extra,
        )
        print(f"Early stopping: {epochs_without_improvement}/{patience}")
        if epochs_without_improvement >= patience:
            print(f"Stopping early. Best validation F1: {best_f1:.4f}")
            break

    if have_best:
        # Mutates the existing teacher, so the existing prediction cells use best weights.
        load_checkpoint(tag=best_tag, model=teacher, optimizer=optimizer)
        print(f"Restored best model from {best_tag} (F1={best_f1:.4f})")
    teacher.eval()
    return teacher, history


In [43]:
teacher, history = train_teacher(
    teacher=teacher,
    train_loader = train_loader,
    val_loader=val_loader,
    label2id=label2id,
    optimizer=optimizer,
    trainable_params=trainable_params,
    tag="encdec_lorma_dora",
    num_epochs=20,
    patience=21,
    min_delta=0.0,
)

Validation prediction: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]



Epoch 1/20
Train Loss:    0.6715
Val Loss:      0.4552
Val Strict F1:  0.0000


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/encdec_lorma_dora_best/optimizer.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (398 outputs left out)

(…)ints/encdec_lorma_dora_best/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/encdec_lorma_dora_best/rng_state.pt -> /tmp/tmp8bf24da8/checkpoints/encdec_lorma_dora_best/rng_state.pt


(…)cdec_lorma_dora_best/training_state.json:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Downloaded: checkpoints/encdec_lorma_dora_best/training_state.json -> /tmp/tmp8bf24da8/checkpoints/encdec_lorma_dora_best/training_state.json
Restored best model from encdec_lorma_dora_best (F1=0.2785)


In [44]:
import json
import zipfile
import os
import torch
from tqdm import tqdm


def generate_predictions(
    teacher,
    processed_test,
    id2label,
    tag,
    max_seq_len=512,
):
    teacher.eval()

    predictions = {}

    with torch.no_grad():

        for doc_id, example in tqdm(
            processed_test.items(),
            desc="Generating predictions"
        ):

            chunks = make_chunks(
                example,
                max_seq_len
            )

            document_entities = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                document_entities.extend(
                    entities
                )

            predictions[doc_id] = document_entities

    # =========================
    # Save predictions.json
    # =========================

    json_path = f"/kaggle/working/predictions.json"

    with open(json_path, "w") as f:
        json.dump(
            predictions,
            f,
            indent=2
        )

    # =========================
    # Zip as tag.zip
    # =========================

    zip_path = f"/kaggle/working/{tag}.zip"

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(
            json_path,
            arcname="prediction.json"
        )

    print(f"Saved: {zip_path}")

    return zip_path

In [45]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Test/fold3-text.json",
    repo_type="dataset",
)

with open(test_file_path, "r") as f:
    fold3_text = json.load(f)

print(f"Loaded {len(fold3_text)} test documents")
processed_test = prepare_lner_data(
    fold3_text,
    None,
    tokenizer
)

print(f"Processed {len(processed_test)} documents")

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Loaded 35 test documents
Processed 35 documents


In [46]:
generate_predictions(
    teacher,
    processed_test,
    id2label,
    "encdec_lorma_dora",
    max_seq_len=512,
)

Generating predictions: 100%|██████████| 35/35 [00:34<00:00,  1.01it/s]

Saved: /kaggle/working/encdec_lorma_dora.zip


'/kaggle/working/encdec_lorma_dora.zip'